In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

...

3.0.0/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [6]:
print(f"Features: {dataset['train'].column_names}")

Features: ['article', 'highlights', 'id']


In [7]:
sample = dataset["train"][0]

In [8]:
sample.keys()

dict_keys(['article', 'highlights', 'id'])

In [9]:
print(f"""Article (excerpt of 500 characters, total length: {len(sample["article"])})""")
# let's 500 characters of article
print(sample["article"][:500])

Article (excerpt of 500 characters, total length: 2527)
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s


In [10]:
print(f"Summary (length: {len(sample["highlights"])}):")
print(sample["highlights"])

Summary (length: 217):
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


## **Text Summarization Pipelines**
- comparing different models over the same input text

In [11]:
sample_text = dataset["train"][1]["article"][:2000] # would be used as input text for models
summaries = {} # the generated summaries by the model would be stored in this dict

In [12]:
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt_tab")
string = "The U.S. is a country. The U.N. is an organization."
sent_tokenize(string)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['The U.S. is a country.', 'The U.N. is an organization.']

### **Baseline Summarization**

- This is one of the simplest baselines in NLP, just choose the first 3 sentences

In [19]:
def three_sentence_summary(text):
  return "\n".join(sent_tokenize(text)[:3])

In [20]:
summaries["baseline"] = three_sentence_summary(sample_text)

In [21]:
summaries['baseline']

'Editor\'s note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events.\nHere, Soledad O\'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.\nMIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor."'

## **GPT-2**

In [25]:
from transformers import pipeline, set_seed
set_seed(42)
pipe = pipeline("text-generation", model="gpt2-xl")
gpt2_query = sample_text + "\nTL;DR:\n"
pipe_out = pipe(gpt2_query, max_length=512, clean_up_tokenization_spaces=True)
summaries["gpt2"] = "\n".join(
 sent_tokenize(pipe_out[0]["generated_text"][len(gpt2_query) :]))

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [26]:
summaries["gpt2"]

"The mentally ill aren't being treated properly and are being housed in an environment that is not conducive to their care.\nThe inmates are living on the ninth floor of the jail, a place where there are no beds and no proper ventilation.\nThe mentally ill inmates are not getting any help and are instead getting worse.\nThe mentally ill inmates are often violent at times, but it could be for a variety of reasons.\nThe inmates are housed in a facility that is not designed for the mentally ill and is inhumane.\nThe number of mentally ill people in the country is increasing and not being adequately treated.\nThe mentally ill are dying in jail and the number of mentally ill people dying is rising.\nThe mentally ill are being put into jails and prisons that are unsafe to them and they are dying in jail.\nThe mentally ill are suffering in jail and hospitals.\nThe number of mentally ill people in jails and prisons is increasing but they are being neglected.\nThe mentally ill are dying in jail

## **T5**

In [27]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-large")

inputs = tokenizer(
    "summarize: " + sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=512
)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
summaries["t5"] = "\n".join(sent_tokenize(summary))

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.95GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [28]:
summaries['t5']

'mentally ill inmates are housed on the "forgotten floor" in Miami-dade county jail .\njudge says many of the inmates are there to face drug charges, assault charges .\njudge says many of the mentally ill are there to avoid confrontations with police .'

## **Bart**

In [29]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

inputs = tokenizer(
    sample_text,          # No "summarize:" prefix for BART
    return_tensors="pt",
    truncation=True,
    max_length=1024       # BART accepts up to 1024 input tokens
)

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    num_beams=4,
    early_stopping=True
)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
summaries["bart"] = "\n".join(sent_tokenize(summary))

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [30]:
summaries['bart']

'Mentally ill inmates are housed on the "forgotten floor" of Miami-Dade jail.\nMost often, they face drug charges or charges of assaulting an officer.\nJudge Steven Leifman says the arrests often result from confrontations with police.\nHe says about one-third of all people in the county jails are mentally ill.'

## **Pegasus**

In [31]:
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")
model = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-xsum")

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

summaries["pegasus"] = "\n".join(sent_tokenize(summary))

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [32]:
summaries['pegasus']

'An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.'

## Comparing different summaries

In [33]:
print("GROUND TRUTH")
print(dataset["train"][1]["highlights"])
print("")
for model_name in summaries:
 print(model_name.upper())
 print(summaries[model_name])
 print("")

GROUND TRUTH
Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While CNN tours facility, patient shouts: "I am the son of the president"
Leifman says the system is unjust and he's fighting for change .

BASELINE
Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events.
Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.
MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor."

GPT2
The mentally ill aren't being treated properly and are being housed in an environment that is not conducive to their care.
The inmates are living on the ninth floor of the jail, a place where there are no beds and no p

## **Bleu**

In [22]:
import evaluate

bleu_metric = evaluate.load("sacrebleu")


In [23]:
import pandas as pd
import numpy as np
bleu_metric.add(
 prediction="the the the the the the", reference=["the cat is on the mat"])
results = bleu_metric.compute(smooth_method="floor", smooth_value=0)
results["precisions"] = [np.round(p, 2) for p in results["precisions"]]
pd.DataFrame.from_dict(results, orient="index", columns=["Value"])


,Value
score,0.0
counts,"[2, 0, 0, 0]"
totals,"[6, 5, 4, 3]"
precisions,"[33.33, 0.0, 0.0, 0.0]"
bp,1.0
sys_len,6
ref_len,6


In [24]:
bleu_metric.add(
 prediction="the cat is on mat", reference=["the cat is on the mat"])
results = bleu_metric.compute(smooth_method="floor", smooth_value=0)
results["precisions"] = [np.round(p, 2) for p in results["precisions"]]
pd.DataFrame.from_dict(results, orient="index", columns=["Value"])

,Value
score,57.893007
counts,"[5, 3, 2, 1]"
totals,"[5, 4, 3, 2]"
precisions,"[100.0, 75.0, 66.67, 50.0]"
bp,0.818731
sys_len,5
ref_len,6


## **Rouge**

In [34]:
import evaluate
import pandas as pd

rouge_metric = evaluate.load("rouge")

reference = dataset["train"][1]["highlights"]

records = []

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

for model_name in summaries:
    rouge_metric.add(
        prediction=summaries[model_name],
        reference=reference
    )

    score = rouge_metric.compute()

    rouge_dict = {
        rn: score[rn]
        for rn in rouge_names
    }

    records.append(rouge_dict)

pd.DataFrame.from_records(
    records,
    index=summaries.keys()
)

,rouge1,rouge2,rougeL,rougeLsum
baseline,0.365079,0.145161,0.206349,0.285714
gpt2,0.159420,0.036496,0.130435,0.159420
t5,0.439560,0.224719,0.329670,0.373626
bart,0.475248,0.222222,0.316832,0.415842
pegasus,0.328358,0.246154,0.179104,0.208955


## **Evaluating PEGASUS on the CNN/DailyMail Dataset**

In [35]:
def evaluate_summaries_baseline(dataset, metric,
 column_text="article",
 column_summary="highlights"):
 summaries = [three_sentence_summary(text) for text in dataset[column_text]]
 metric.add_batch(predictions=summaries,
 references=dataset[column_summary])
 score = metric.compute()
 return score

In [36]:
test_sampled = dataset["test"].shuffle(seed=42).select(range(1000))

score = evaluate_summaries_baseline(test_sampled, rouge_metric)

rouge_dict = {rn: score[rn] for rn in rouge_names}

pd.DataFrame.from_dict(
    rouge_dict,
    orient="index",
    columns=["baseline"]
).T

,rouge1,rouge2,rougeL,rougeLsum
baseline,0.389276,0.171296,0.245061,0.354239


In [37]:
from tqdm import tqdm
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
def chunks(list_of_elements, batch_size):
  #Yield successive batch-sized chunks from list_of_elements.
   for i in range(0, len(list_of_elements), batch_size):
    yield list_of_elements[i : i + batch_size]


In [40]:
def evaluate_summaries_pegasus(dataset,metric,model,tokenizer,batch_size= 16,device = device,column_text = "article",column_summary="highlights"):
  article_batches = list(chunks(dataset[column_text],batch_size))
  target_batches = list(chunks(dataset[column_summary],batch_size))

  for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total = len(article_batches)):

    inputs = tokenizer(article_batch, max_length = 1024, truncation = True, padding = "max_length", return_tensors = "pt")

    summaries = model.generate(input_ids = inputs['input_ids'].to(device),attention_mask =inputs["attention_mask"].to(device), length_penalty = 0.8, num_beams = 8, max_length = 128)

    decoded_summaries = [tokenizer.decode(s, skip_special_tokens = True, clean_up_tokenization_spaces = True) for s in summaries]

    decoded_summaries = [d.replace("<n>", " ") for d in decoded_summaries]
    metric.add_batch(predictions = decoded_summaries, refrences = target_batch)

  score = metric.compute()
  return score


In [41]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)
score = evaluate_summaries_pegasus(test_sampled, rouge_metric,
 model, tokenizer, batch_size=8)
rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)
pd.DataFrame(rouge_dict, index=["pegasus"])

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/125 [00:04<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 213.81 MiB is free. Including non-PyTorch memory, this process has 14.35 GiB memory in use. Of the allocated memory 14.20 GiB is allocated by PyTorch, and 24.04 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)